In [10]:
#import library
import pandas as pd

# load all csv files
customers = pd.read_csv("customer.csv")
orders = pd.read_csv("orders.csv")
products = pd.read_csv("products.csv")
payments = pd.read_csv("payment.csv")

# rows and columns count
print(customers.shape)
print(orders.shape)
print(products.shape)
print(payments.shape)

(8, 5)
(8, 6)
(6, 5)
(8, 4)


In [11]:
# Q2+
print(customers.columns)
print(products.columns)
print(orders.columns)
print(payments.columns)


Index(['customer_id', 'customer_name', 'city', 'state', 'segment'], dtype='object')
Index(['product_id', 'product_name', 'category', 'sub_category', 'unit_price'], dtype='object')
Index(['order_id', 'order_date', 'customer_id', 'product_id', 'quantity',
       'region'],
      dtype='object')
Index(['payment_id', 'order_id', 'payment_mode', 'payment_status'], dtype='object')


In [12]:
# Q3
orders["order_date"] = pd.to_datetime(orders["order_date"])
print(orders.dtypes)


order_id               object
order_date     datetime64[ns]
customer_id            object
product_id             object
quantity                int64
region                 object
dtype: object


In [13]:
# Q4
south_orders = orders[orders["region"] == "South"]


In [14]:
# Q5
corporate_customers = customers[customers["segment"] == "Corporate"]


In [15]:
# Q6
orders[orders["quantity"] > 10][["order_id", "customer_id", "quantity"]]


,order_id,customer_id,quantity
2,O9003,C003,25
5,O9006,C006,15
7,O9008,C008,20


In [16]:
# Q7
orders.groupby("product_id")["quantity"].sum().sort_values(ascending=False)


product_id
P102    45
P101    25
P106     3
P103     2
P104     1
P105     1
Name: quantity, dtype: int64

In [17]:
# Q8
orders.groupby("region")["order_id"].count().reset_index(name="order_count")


,region,order_count
0,North,2
1,South,4
2,West,2


In [18]:
# Q9
products.groupby("category")["unit_price"].mean()


category
Furniture          6350.0
Office Supplies      27.5
Technology         9575.0
Name: unit_price, dtype: float64

In [19]:
# Q10
orders.merge(customers, on="customer_id", how="inner")[
    ["order_id", "customer_name", "city", "region"]
]


,order_id,customer_name,city,region
0,O9001,Rahul Sharma,Bengaluru,South
1,O9002,Anita Verma,Delhi,North
2,O9003,Suresh Iyer,Chennai,South
3,O9004,Priya Nair,Kochi,South
4,O9005,Amit Patel,Ahmedabad,West
5,O9006,Neha Singh,Jaipur,North
6,O9007,Ravi Kumar,Hyderabad,South
7,O9008,Meera Joshi,Pune,West


In [30]:
# Q11
op = orders.merge(products, on="product_id", how="inner")
op["order_value"] = op["quantity"] * op["unit_price"]



In [21]:
# Q12
full = orders.merge(customers, on="customer_id").merge(products, on="product_id")
full["order_value"] = full["quantity"] * full["unit_price"]
full[["order_id", "customer_name", "product_name", "category", "quantity", "order_value"]]


,order_id,customer_name,product_name,category,quantity,order_value
0,O9001,Rahul Sharma,Notebook,Office Supplies,10,450
1,O9002,Anita Verma,Printer,Technology,1,18500
2,O9003,Suresh Iyer,Pen,Office Supplies,25,250
3,O9004,Priya Nair,Office Chair,Furniture,2,11000
4,O9005,Amit Patel,Study Table,Furniture,1,7200
5,O9006,Neha Singh,Notebook,Office Supplies,15,675
6,O9007,Ravi Kumar,USB Drive,Technology,3,1950
7,O9008,Meera Joshi,Pen,Office Supplies,20,200


In [22]:
# Q13
opc = full.merge(payments, on="order_id", how="inner")
opc[opc["payment_status"] == "Pending"]


,order_id,order_date,customer_id,product_id,quantity,region,customer_name,city,state,segment,product_name,category,sub_category,unit_price,order_value,payment_id,payment_mode,payment_status
3,O9004,2024-01-15,C004,P103,2,South,Priya Nair,Kochi,Kerala,Home Office,Office Chair,Furniture,Chairs,5500,11000,PAY04,Debit Card,Pending
7,O9008,2024-01-25,C008,P102,20,West,Meera Joshi,Pune,Maharashtra,Home Office,Pen,Office Supplies,Writing,10,200,PAY08,UPI,Pending


In [23]:
# Q14
opc[opc["payment_status"] == "Completed"] \
    .groupby("region")["order_value"].sum()


region
North    19175
South     2650
West      7200
Name: order_value, dtype: int64

In [24]:
# Q15
opc[opc["payment_status"] == "Completed"] \
    .groupby("state") \
    .agg(total_orders=("order_id", "count"),
         total_revenue=("order_value", "sum"))


,total_orders,total_revenue
state,,
Delhi,1,18500
Gujarat,1,7200
Karnataka,1,450
Rajasthan,1,675
Tamil Nadu,1,250
Telangana,1,1950


In [25]:
# Q16
cust_val = opc[opc["payment_status"] == "Completed"] \
    .groupby(["customer_name", "city"])["order_value"].sum().reset_index()

cust_val[cust_val["order_value"] > 20000]


,customer_name,city,order_value


In [26]:
# Q17
payments["payment_mode"].value_counts()


payment_mode
UPI            4
Credit Card    2
Debit Card     1
Net Banking    1
Name: count, dtype: int64

In [27]:
# Q18
rp = opc.groupby(["region", "product_name"])["quantity"].sum().reset_index()
rp.loc[rp.groupby("region")["quantity"].idxmax()]


,region,product_name,quantity
0,North,Notebook,15
4,South,Pen,25
6,West,Pen,20


In [28]:
# Q19
opc[opc["payment_status"] == "Pending"][
    ["product_name", "order_id", "payment_status"]
]


,product_name,order_id,payment_status
3,Office Chair,O9004,Pending
7,Pen,O9008,Pending


In [29]:
# Q20
opc.groupby("region").agg(
    total_orders=("order_id", "count"),
    completed_orders=("payment_status", lambda x: (x == "Completed").sum()),
    pending_orders=("payment_status", lambda x: (x == "Pending").sum()),
    total_revenue=("order_value", lambda x: x[opc.loc[x.index, "payment_status"] == "Completed"].sum())
)


,total_orders,completed_orders,pending_orders,total_revenue
region,,,,
North,2,2,0,19175
South,4,3,1,2650
West,2,1,1,7200
